# Probability and Statistics MT-2005
## Project 2
### Classification through Naive Bayes and Decision Trees

Instructions:

Run all cells and be sure to read the comments as they may contain instructions for you. If you are required to answer a question or to make a comment, then please do it in a new markdown at the end of the file.


Submission guidelines: After execution, please download the notebook and upload it on GCR, before the deadline.

Naming convention: section_i24xxxx

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
x = False #set this variable to true to proceed

In [ ]:
if x:
  # Load Dataset
  df = pd.read_csv("mushrooms.csv")

In [ ]:
df.replace("?", np.nan, inplace=True)
df.dropna(inplace=True) #missing values will not be missed

In [ ]:
print(df['class'].value_counts())
sns.countplot(x='class', data=df)
plt.title("Class Distribution (e = edible, p = poisonous)")
plt.show()

In [ ]:
# creating train test splits
X = df.drop("class", axis=1)
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
#Random state = 42 is a very common occurence in ML and AI. But rarely do people know why we use this particular value.
#Create a markdown at the end of the file and answer why this value is used. Label this markdown as 'Random State'

### Manual Naive Bayes Implementation

In [ ]:
# Step 1: Compute Priors
priors = y_train.value_counts(normalize=True)

# Step 2: Compute Likelihoods
likelihoods = {}

for col in X_train.columns:
    likelihoods[col] = {}
    for cls in y_train.unique():
        subset = X_train[y_train == cls]
        probs = subset[col].value_counts(normalize=True)
        likelihoods[col][cls] = probs

# Step 3: Prediction Function
def manual_nb_predict(row):
    scores = {}

    for cls in priors.index:
        score = priors[cls]

        for col in row.index:
            value = row[col]

            if value in likelihoods[col][cls]:
                score *= likelihoods[col][cls][value]
            else:
                score *= 1e-6

        scores[cls] = score

    return max(scores, key=scores.get)

# Step 4: Predict Test Set
y_pred_nb = X_test.apply(manual_nb_predict, axis=1)

print("Manual Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

In [ ]:
def entropy(y):
    values, counts = np.unique(y, return_counts=True)
    probs = counts / counts.sum()
    return -np.sum(probs * np.log2(probs))
print("Entropy of entire dataset:", entropy(y_train))

What does this particular value of entropy tell us? Label this cell as 'Entropy'

In [ ]:
def information_gain(X, y, feature):
    parent_entropy = entropy(y)

    values = X[feature].unique()

    weighted_entropy = 0

    for val in values:
        subset_y = y[X[feature] == val]
        weight = len(subset_y) / len(y)
        weighted_entropy += weight * entropy(subset_y)

    return parent_entropy - weighted_entropy

features = X_train.columns
for feature in features:
    ig = information_gain(X_train, y_train, feature)
    print(f"Information Gain for {feature}: {ig:.4f}")

Are there any information gain values that are zero? If yes then discuss why this could happen and what does it tell us.

Label this cell as Information Gain.

### Decision Tree

In [ ]:
X_combined = pd.concat([X_train, X_test], ignore_index=True)
X_combined_encoded = pd.get_dummies(X_combined)

X_train_encoded = X_combined_encoded.iloc[:len(X_train)]
X_test_encoded = X_combined_encoded.iloc[len(X_train):]

dt_entropy = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=4,
    random_state=42)

dt_entropy.fit(X_train_encoded, y_train)
y_pred_entropy = dt_entropy.predict(X_test_encoded)
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_entropy))

Based on what you have studied in class, make a comment on why the accuracies of these models vary.

Label this cell as 'Accuracy Difference'

The values of train test split can be changed to give varying results. You are required to use 3-4 values of the test split, for each of the models, and observe the difference in outputs.

Enter your answer at the end of the file in a new markdown. Label this as 'Train Test Split'

Which of the classifiers is more suited for the datset that we have used ?

Label this markdown as 'Classifier choice'

Visualising the classifications?

The designer of this project had a questionable moment of brilliance and he thought that 'why not visualise the classification?'; but soon he realised his error. :)

Explain why visualisation is impractical and hard to interpret in this scenario.

Label this cell as 'Visualisation'